# Runbook de auditoria (somente leitura)

Este notebook não altera arquivos. Ele apenas coleta evidências do estado atual do monorepo.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / 'package.json').exists() and (base / 'apps').exists():
            return base
    return start

ROOT = find_repo_root(Path.cwd().resolve())
print('Repo root:', ROOT)

## 1) Presença de arquivos e pastas-chave

In [ ]:
key_paths = [
    'package.json',
    'nest-cli.json',
    'tsconfig.json',
    'apps',
    'packages',
    'docs',
    'spec',
    '.github/instructions',
    '.github/prompts',
    '.github/skills',
]
for rel in key_paths:
    p = ROOT / rel
    print(('✅' if p.exists() else '❌'), rel)

## 2) Apps detectados e projetos do Nest

In [ ]:
apps_dir = ROOT / 'apps'
apps = sorted([p.name for p in apps_dir.iterdir() if p.is_dir()]) if apps_dir.exists() else []
print('Apps encontrados:', apps)

nest_cli_path = ROOT / 'nest-cli.json'
if nest_cli_path.exists():
    nest_cli = json.loads(nest_cli_path.read_text(encoding='utf-8'))
    projects = sorted((nest_cli.get('projects') or {}).keys())
    print('Projetos no nest-cli.json:', projects)
else:
    print('❌ nest-cli.json não encontrado')

## 3) Health + main.ts por app (somente leitura)

In [ ]:
target_apps = ['users-api', 'llm-ops-api', 'sharepoint-api', 'sync-api', 'monorepo-ai-llm']
for app in target_apps:
    src = ROOT / 'apps' / app / 'src'
    if not src.exists():
        print('❌', app, '-> src ausente')
        continue

    health = src / 'health.controller.ts'
    app_module = src / 'app.module.ts'
    main_ts = src / 'main.ts'

    hm = health.exists()
    am_ok = False
    main_ok = False

    if app_module.exists():
        s = app_module.read_text(encoding='utf-8')
        am_ok = ('./health.controller' in s) and ('HealthController' in s)

    if main_ts.exists():
        s = main_ts.read_text(encoding='utf-8')
        main_ok = 'process.env.PORT' in s

    print(('✅' if (hm and am_ok and main_ok) else '⚠️'), app, '| health:', hm, '| app.module:', am_ok, '| main(PORT):', main_ok)

## 4) Governança (somente leitura)

In [ ]:
checks = [
    'spec/fases/fase-03-spec-packaging-boundaries.md',
    'spec/fases/fase-01-arquitetura-blueprint.md',
    'spec/fases/fase-02-plano-convencoes-naming.md',
    '.github/skills/microservice-guardian',
    'spec/execucao/fase-01-conventions-bootstrap.ipynb',
    'spec/execucao/fase-01-nest-monorepo-bootstrap-exec.ipynb',
    'spec/testes/fase-02-validacao-config-estado-atual.ipynb',
]
for rel in checks:
    print(('✅' if (ROOT / rel).exists() else '❌'), rel)